In [1]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

df = pd.read_csv("data/train.csv")

X = df.drop(columns=["id", "addicted_label"]).copy()
y = df["addicted_label"]
cat_cols = ["gender", "stress_level", "academic_work_impact"]
for c in cat_cols:
    X[c] = X[c].astype("category")

params = dict(
    objective="binary", metric="auc",
    learning_rate=0.05, num_leaves=63, min_child_samples=100,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
    verbose=-1, n_jobs=-1,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(y))
best_iters = []

for fold, (tr, va) in enumerate(cv.split(X, y)):
    dtr = lgb.Dataset(X.iloc[tr], y.iloc[tr])
    dva = lgb.Dataset(X.iloc[va], y.iloc[va])
    m = lgb.train(
        params, dtr, num_boost_round=3000, valid_sets=[dva],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
    )
    oof[va] = m.predict(X.iloc[va], num_iteration=m.best_iteration)
    best_iters.append(m.best_iteration)
    print(f"fold {fold}: best_iter={m.best_iteration}  auc={roc_auc_score(y.iloc[va], oof[va]):.5f}")

print(f"\nOOF AUC: {roc_auc_score(y, oof):.5f}")
print(f"mean best_iter: {int(np.mean(best_iters))}")

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1777]	valid_0's auc: 0.962989
fold 0: best_iter=1777  auc=0.96299
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1509]	valid_0's auc: 0.9637
fold 1: best_iter=1509  auc=0.96370
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1490]	valid_0's auc: 0.964122
fold 2: best_iter=1490  auc=0.96412
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1679]	valid_0's auc: 0.964629
fold 3: best_iter=1679  auc=0.96463
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1574]	valid_0's auc: 0.963437
fold 4: best_iter=1574  auc=0.96344

OOF AUC: 0.96377
mean best_iter: 1605


In [4]:
n_best = int(np.mean(best_iters))
final = lgb.LGBMClassifier(**{k: v for k, v in params.items() if k != "metric"},
                           n_estimators=n_best)
final.fit(X, y)

test = pd.read_csv("data/test.csv")
test_ids = test["id"]
X_test = test.drop(columns=["id"]).copy()
for c in cat_cols:
    X_test[c] = X_test[c].astype("category")

proba = final.predict_proba(X_test)[:, 1]
pd.DataFrame({"id": test_ids, "addicted_label": proba}).to_csv("submission.csv", index=False)